In [ ]:
!nvidia-smi

In [ ]:
import os
import pickle
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

In [ ]:
!wget -O WESAD.zip https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download

In [ ]:
!ls -lh WESAD.zip

In [ ]:
import os
import zipfile

ZIP_PATH = "/content/WESAD.zip"
EXTRACT_PATH = "/content/WESAD"

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Extraction complete.")

In [ ]:
!du -sh /content/WESAD
!find /content/WESAD -maxdepth 2 -type d | head -30

In [ ]:
import glob
import os

pkl_files = sorted(
    glob.glob(
        "/content/WESAD/**/S*.pkl",
        recursive=True
    )
)

print("Number of PKL files:", len(pkl_files))

for f in pkl_files:
    print(
        os.path.basename(f),
        "->",
        f
    )

In [ ]:
import pickle

s2_candidates = [
    f for f in pkl_files
    if os.path.basename(f) == "S2.pkl"
]

assert len(s2_candidates) == 1, (
    f"Expected exactly one S2.pkl, found {len(s2_candidates)}"
)

S2_PATH = s2_candidates[0]

print("Loading:")
print(S2_PATH)

with open(S2_PATH, "rb") as f:
    s2 = pickle.load(
        f,
        encoding="latin1"
    )

print("\nTop-level keys:")
print(s2.keys())

In [ ]:
print("Subject:", s2.get("subject"))

print("\nSignal devices:")
print(s2["signal"].keys())

print("\nChest modalities:")
print(s2["signal"]["chest"].keys())

print("\nWrist modalities:")
print(s2["signal"]["wrist"].keys())

print("\nLabel shape:")
print(s2["label"].shape)

print("\nUnique labels:")
print(
    np.unique(
        s2["label"],
        return_counts=True
    )
)

In [ ]:
print("\n===== CHEST =====")

for key, value in s2["signal"]["chest"].items():
    print(
        f"{key:10s}",
        np.asarray(value).shape
    )


print("\n===== WRIST =====")

for key, value in s2["signal"]["wrist"].items():
    print(
        f"{key:10s}",
        np.asarray(value).shape
    )

In [ ]:
# ==========================================
# T1.2 preprocessing configuration
# ==========================================

CHEST_FS = 700
BVP_FS   = 64
ACC_FS   = 32

WINDOW_SEC = 30
STEP_SEC   = 30

# WESAD:
# 1 = baseline
# 2 = stress
KEEP_LABELS = [1, 2]

# Binary mapping
# baseline -> 0
# stress   -> 1
LABEL_MAP = {
    1: 0,
    2: 1
}

print("Window:", WINDOW_SEC, "seconds")
print("Step:", STEP_SEC, "seconds")
print("Task: baseline (0) vs stress (1)")

In [ ]:
from scipy.stats import skew, kurtosis
import numpy as np


def safe_stats(x):

    x = np.asarray(
        x,
        dtype=np.float32
    ).reshape(-1)

    x = x[
        np.isfinite(x)
    ]

    if len(x) == 0:
        return [0.0] * 10

    mean = np.mean(x)
    std = np.std(x)

    minimum = np.min(x)
    maximum = np.max(x)

    median = np.median(x)

    q25 = np.percentile(x, 25)
    q75 = np.percentile(x, 75)

    rms = np.sqrt(
        np.mean(x ** 2)
    )

    sk = (
        skew(x)
        if std > 1e-8
        else 0.0
    )

    ku = (
        kurtosis(x)
        if std > 1e-8
        else 0.0
    )

    return [
        mean,
        std,
        minimum,
        maximum,
        median,
        q25,
        q75,
        rms,
        sk,
        ku
    ]

In [ ]:
from scipy.signal import find_peaks


def ecg_features(x, fs=700):

    x = np.asarray(
        x,
        dtype=np.float32
    ).reshape(-1)

    features = safe_stats(x)

    # Basic peak-based HR estimate
    try:

        distance = int(
            0.35 * fs
        )

        prominence = max(
            np.std(x) * 0.5,
            1e-6
        )

        peaks, _ = find_peaks(
            x,
            distance=distance,
            prominence=prominence
        )

        if len(peaks) >= 2:

            rr = (
                np.diff(peaks)
                / fs
            )

            mean_rr = np.mean(rr)
            std_rr = np.std(rr)

            hr = (
                60.0 / mean_rr
                if mean_rr > 0
                else 0.0
            )

        else:

            mean_rr = 0.0
            std_rr = 0.0
            hr = 0.0

    except Exception:

        mean_rr = 0.0
        std_rr = 0.0
        hr = 0.0

    features.extend([
        mean_rr,
        std_rr,
        hr
    ])

    return features

In [ ]:
def bvp_features(x):

    return safe_stats(x)

In [ ]:
def acc_features(x):

    x = np.asarray(
        x,
        dtype=np.float32
    )

    features = []

    # X, Y, Z
    for axis in range(3):

        features.extend(
            safe_stats(
                x[:, axis]
            )
        )

    magnitude = np.sqrt(
        np.sum(
            x ** 2,
            axis=1
        )
    )

    features.extend(
        safe_stats(
            magnitude
        )
    )

    return features

In [ ]:
def process_subject(
    pkl_path,
    window_sec=30,
    step_sec=30
):

    with open(
        pkl_path,
        "rb"
    ) as f:

        data = pickle.load(
            f,
            encoding="latin1"
        )

    subject = data["subject"]

    labels = np.asarray(
        data["label"]
    )

    ecg = np.asarray(
        data["signal"]["chest"]["ECG"]
    ).reshape(-1)

    bvp = np.asarray(
        data["signal"]["wrist"]["BVP"]
    ).reshape(-1)

    acc = np.asarray(
        data["signal"]["wrist"]["ACC"]
    )

    # Determine common recording duration
    durations = [
        len(labels) / CHEST_FS,
        len(ecg) / CHEST_FS,
        len(bvp) / BVP_FS,
        len(acc) / ACC_FS
    ]

    duration_sec = min(
        durations
    )

    rows = []

    start_sec = 0

    while (
        start_sec + window_sec
        <= duration_sec
    ):

        end_sec = (
            start_sec
            + window_sec
        )

        # ----------------------------------
        # Label indices
        # ----------------------------------

        label_start = int(
            start_sec * CHEST_FS
        )

        label_end = int(
            end_sec * CHEST_FS
        )

        label_window = labels[
            label_start:label_end
        ]

        # Only retain windows whose samples
        # are entirely baseline or stress.
        valid_mask = np.isin(
            label_window,
            KEEP_LABELS
        )

        valid_fraction = (
            valid_mask.mean()
        )

        # Require >=95% relevant labels
        if valid_fraction < 0.95:

            start_sec += step_sec
            continue

        valid_labels = label_window[
            valid_mask
        ]

        values, counts = np.unique(
            valid_labels,
            return_counts=True
        )

        majority_label = values[
            np.argmax(counts)
        ]

        purity = (
            counts.max()
            / len(valid_labels)
        )

        # Avoid transition windows
        if purity < 0.95:

            start_sec += step_sec
            continue

        if majority_label not in LABEL_MAP:

            start_sec += step_sec
            continue

        y = LABEL_MAP[
            int(majority_label)
        ]

        # ----------------------------------
        # ECG window
        # ----------------------------------

        ecg_start = int(
            start_sec * CHEST_FS
        )

        ecg_end = int(
            end_sec * CHEST_FS
        )

        ecg_window = ecg[
            ecg_start:ecg_end
        ]

        # ----------------------------------
        # BVP window
        # ----------------------------------

        bvp_start = int(
            start_sec * BVP_FS
        )

        bvp_end = int(
            end_sec * BVP_FS
        )

        bvp_window = bvp[
            bvp_start:bvp_end
        ]

        # ----------------------------------
        # ACC window
        # ----------------------------------

        acc_start = int(
            start_sec * ACC_FS
        )

        acc_end = int(
            end_sec * ACC_FS
        )

        acc_window = acc[
            acc_start:acc_end
        ]

        # Safety check
        if (
            len(ecg_window)
            != window_sec * CHEST_FS
            or
            len(bvp_window)
            != window_sec * BVP_FS
            or
            len(acc_window)
            != window_sec * ACC_FS
        ):

            start_sec += step_sec
            continue

        # ----------------------------------
        # Extract modality features
        # ----------------------------------

        ef = ecg_features(
            ecg_window,
            CHEST_FS
        )

        bf = bvp_features(
            bvp_window
        )

        af = acc_features(
            acc_window
        )

        row = {
            "subject": subject,
            "start_sec": start_sec,
            "label": y
        }

        for i, value in enumerate(ef):
            row[f"ECG_{i}"] = value

        for i, value in enumerate(bf):
            row[f"BVP_{i}"] = value

        for i, value in enumerate(af):
            row[f"ACC_{i}"] = value

        rows.append(row)

        start_sec += step_sec

    return pd.DataFrame(rows)

In [ ]:
s2_df = process_subject(
    S2_PATH,
    window_sec=WINDOW_SEC,
    step_sec=STEP_SEC
)

print("Shape:", s2_df.shape)

print("\nLabel distribution:")
print(
    s2_df["label"]
    .value_counts()
    .sort_index()
)

print("\nSubjects:")
print(
    s2_df["subject"]
    .unique()
)

print("\nFirst rows:")
print(
    s2_df[
        [
            "subject",
            "start_sec",
            "label"
        ]
    ].head(10)
)

print(
    "\nMissing values:",
    s2_df.isna().sum().sum()
)

print(
    "Infinite numeric values:",
    np.isinf(
        s2_df.select_dtypes(
            include=np.number
        ).values
    ).sum()
)

In [ ]:
import gc
import os

FEATURE_DIR = "/content/WESAD_features"
os.makedirs(FEATURE_DIR, exist_ok=True)

all_subject_dfs = []

for idx, pkl_path in enumerate(pkl_files, 1):

    subject_name = os.path.basename(
        pkl_path
    ).replace(".pkl", "")

    print(
        f"[{idx}/{len(pkl_files)}] "
        f"Processing {subject_name}..."
    )

    subject_df = process_subject(
        pkl_path,
        window_sec=WINDOW_SEC,
        step_sec=STEP_SEC
    )

    # Save subject immediately
    save_path = os.path.join(
        FEATURE_DIR,
        f"{subject_name}_features.csv"
    )

    subject_df.to_csv(
        save_path,
        index=False
    )

    print(
        f"   Windows: {len(subject_df)}"
    )

    print(
        "   Baseline:",
        (subject_df["label"] == 0).sum(),
        "| Stress:",
        (subject_df["label"] == 1).sum()
    )

    all_subject_dfs.append(
        subject_df
    )

    gc.collect()

print("\nAll subjects processed.")

In [ ]:
wesad_features = pd.concat(
    all_subject_dfs,
    ignore_index=True
)

print(
    "Combined shape:",
    wesad_features.shape
)

print("\nOverall label distribution:")
print(
    wesad_features[
        "label"
    ].value_counts().sort_index()
)

print("\nWindows per subject:")
print(
    wesad_features.groupby(
        "subject"
    ).size()
)

print(
    "\nMissing:",
    wesad_features.isna()
    .sum().sum()
)

numeric_data = (
    wesad_features
    .select_dtypes(
        include=np.number
    )
)

print(
    "Infinite:",
    np.isinf(
        numeric_data.values
    ).sum()
)

In [ ]:
wesad_features.to_csv(
    "/content/WESAD_T1_2_features.csv",
    index=False
)

print("Feature dataset saved.")

In [ ]:
ECG_COLS = [
    c for c in wesad_features.columns
    if c.startswith("ECG_")
]

BVP_COLS = [
    c for c in wesad_features.columns
    if c.startswith("BVP_")
]

ACC_COLS = [
    c for c in wesad_features.columns
    if c.startswith("ACC_")
]

print("ECG features:", len(ECG_COLS))
print("BVP features:", len(BVP_COLS))
print("ACC features:", len(ACC_COLS))

print(
    "Total:",
    len(ECG_COLS)
    + len(BVP_COLS)
    + len(ACC_COLS)
)

In [ ]:
TRAIN_SUBJECTS = [
    "S2", "S3", "S4",
    "S5", "S6", "S7",
    "S8", "S9", "S10"
]

VAL_SUBJECTS = [
    "S11",
    "S13",
    "S14"
]

TEST_SUBJECTS = [
    "S15",
    "S16",
    "S17"
]


train_df = wesad_features[
    wesad_features["subject"]
    .isin(TRAIN_SUBJECTS)
].copy()

val_df = wesad_features[
    wesad_features["subject"]
    .isin(VAL_SUBJECTS)
].copy()

test_df = wesad_features[
    wesad_features["subject"]
    .isin(TEST_SUBJECTS)
].copy()


print("TRAIN:", train_df.shape)
print(train_df["label"].value_counts())

print("\nVALIDATION:", val_df.shape)
print(val_df["label"].value_counts())

print("\nTEST:", test_df.shape)
print(test_df["label"].value_counts())

In [ ]:
train_subjects = set(
    train_df["subject"].unique()
)

val_subjects = set(
    val_df["subject"].unique()
)

test_subjects = set(
    test_df["subject"].unique()
)

assert train_subjects.isdisjoint(
    val_subjects
)

assert train_subjects.isdisjoint(
    test_subjects
)

assert val_subjects.isdisjoint(
    test_subjects
)

print("\n✓ No subject leakage")

print("Train subjects:", sorted(train_subjects))
print("Val subjects:", sorted(val_subjects))
print("Test subjects:", sorted(test_subjects))

In [ ]:
from sklearn.preprocessing import StandardScaler

ALL_FEATURE_COLS = (
    ECG_COLS
    + BVP_COLS
    + ACC_COLS
)

scaler = StandardScaler()

X_train = scaler.fit_transform(
    train_df[
        ALL_FEATURE_COLS
    ].values
)

X_val = scaler.transform(
    val_df[
        ALL_FEATURE_COLS
    ].values
)

X_test = scaler.transform(
    test_df[
        ALL_FEATURE_COLS
    ].values
)

y_train = train_df[
    "label"
].values.astype(np.int64)

y_val = val_df[
    "label"
].values.astype(np.int64)

y_test = test_df[
    "label"
].values.astype(np.int64)


print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

print()

print(
    "Train stress prevalence:",
    y_train.mean()
)

print(
    "Val stress prevalence:",
    y_val.mean()
)

print(
    "Test stress prevalence:",
    y_test.mean()
)

In [ ]:
ECG_IDX = np.arange(
    0,
    len(ECG_COLS)
)

BVP_IDX = np.arange(
    len(ECG_COLS),
    len(ECG_COLS) + len(BVP_COLS)
)

ACC_IDX = np.arange(
    len(ECG_COLS) + len(BVP_COLS),
    len(ALL_FEATURE_COLS)
)

print("ECG:", ECG_IDX)
print("BVP:", BVP_IDX)
print("ACC:", ACC_IDX)

print("\nCounts:")
print("ECG:", len(ECG_IDX))
print("BVP:", len(BVP_IDX))
print("ACC:", len(ACC_IDX))

In [ ]:
class FeatureDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.y[idx]
        )


train_dataset = FeatureDataset(
    X_train,
    y_train
)

val_dataset = FeatureDataset(
    X_val,
    y_val
)

test_dataset = FeatureDataset(
    X_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print(
    len(train_dataset),
    len(val_dataset),
    len(test_dataset)
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=DEVICE
)

print(
    "Class weights:",
    class_weights
)

In [ ]:
class EarlyFusionNet(nn.Module):

    def __init__(self, input_dim=63):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                64
            ),

            nn.ReLU(),

            nn.Dropout(0.25),

            nn.Linear(
                64,
                32
            ),

            nn.ReLU(),

            nn.Dropout(0.20),

            nn.Linear(
                32,
                2
            )
        )

    def forward(self, x):

        return self.net(x)

In [ ]:
from copy import deepcopy


def train_model(
    model,
    train_loader,
    val_loader,
    epochs=100,
    lr=1e-3,
    patience=15
):

    model = model.to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    best_auc = -np.inf
    best_state = None

    patience_counter = 0

    for epoch in range(1, epochs + 1):

        # ==========================
        # TRAIN
        # ==========================

        model.train()

        total_loss = 0.0

        for xb, yb in train_loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(xb)

            loss = criterion(
                logits,
                yb
            )

            loss.backward()

            optimizer.step()

            total_loss += (
                loss.item()
                * len(yb)
            )

        # ==========================
        # VALIDATION
        # ==========================

        model.eval()

        probs = []
        targets = []

        with torch.no_grad():

            for xb, yb in val_loader:

                xb = xb.to(DEVICE)

                logits = model(xb)

                prob = torch.softmax(
                    logits,
                    dim=1
                )[:, 1]

                probs.extend(
                    prob.cpu().numpy()
                )

                targets.extend(
                    yb.numpy()
                )

        val_auc = roc_auc_score(
            targets,
            probs
        )

        train_loss = (
            total_loss
            / len(train_loader.dataset)
        )

        if val_auc > best_auc:

            best_auc = val_auc

            best_state = deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if (
            epoch == 1
            or epoch % 10 == 0
        ):

            print(
                f"Epoch {epoch:03d} | "
                f"Loss={train_loss:.4f} | "
                f"Val AUROC={val_auc:.4f}"
            )

        if patience_counter >= patience:

            print(
                f"Early stopping at epoch {epoch}"
            )

            break

    model.load_state_dict(
        best_state
    )

    print(
        f"Best validation AUROC: "
        f"{best_auc:.4f}"
    )

    return model

In [ ]:
def evaluate_model(
    model,
    loader,
    name="Model"
):

    model.eval()

    probs = []
    preds = []
    targets = []

    with torch.no_grad():

        for xb, yb in loader:

            xb = xb.to(DEVICE)

            logits = model(xb)

            prob = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            pred = logits.argmax(
                dim=1
            )

            probs.extend(
                prob.cpu().numpy()
            )

            preds.extend(
                pred.cpu().numpy()
            )

            targets.extend(
                yb.numpy()
            )

    probs = np.asarray(probs)
    preds = np.asarray(preds)
    targets = np.asarray(targets)

    results = {

        "Model": name,

        "Accuracy":
            accuracy_score(
                targets,
                preds
            ),

        "Precision":
            precision_score(
                targets,
                preds,
                zero_division=0
            ),

        "Recall":
            recall_score(
                targets,
                preds,
                zero_division=0
            ),

        "F1":
            f1_score(
                targets,
                preds,
                zero_division=0
            ),

        "AUROC":
            roc_auc_score(
                targets,
                probs
            )
    }

    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    for key, value in results.items():

        if key != "Model":
            print(
                f"{key:10s}: "
                f"{value:.4f}"
            )

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            targets,
            preds
        )
    )

    return results

In [ ]:
torch.manual_seed(SEED)

early_model = EarlyFusionNet(
    input_dim=len(
        ALL_FEATURE_COLS
    )
)

early_model = train_model(
    early_model,
    train_loader,
    val_loader,
    epochs=100,
    lr=1e-3,
    patience=15
)

In [ ]:
early_results = evaluate_model(
    early_model,
    test_loader,
    name="Early Fusion"
)

In [ ]:
class ModalityBranch(nn.Module):

    def __init__(
        self,
        input_dim
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                32
            ),

            nn.ReLU(),

            nn.Dropout(0.20),

            nn.Linear(
                32,
                16
            ),

            nn.ReLU(),

            nn.Linear(
                16,
                2
            )
        )

    def forward(self, x):

        return self.net(x)


class LateFusionNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.ecg = ModalityBranch(
            len(ECG_IDX)
        )

        self.bvp = ModalityBranch(
            len(BVP_IDX)
        )

        self.acc = ModalityBranch(
            len(ACC_IDX)
        )


    def forward(self, x):

        ecg_logits = self.ecg(
            x[:, ECG_IDX]
        )

        bvp_logits = self.bvp(
            x[:, BVP_IDX]
        )

        acc_logits = self.acc(
            x[:, ACC_IDX]
        )

        fused_logits = (
            ecg_logits
            + bvp_logits
            + acc_logits
        ) / 3.0

        return fused_logits

In [ ]:
torch.manual_seed(SEED)

late_model = LateFusionNet()

late_model = train_model(
    late_model,
    train_loader,
    val_loader,
    epochs=100,
    lr=1e-3,
    patience=15
)

late_results = evaluate_model(
    late_model,
    test_loader,
    name="Late Fusion"
)

In [ ]:
class ReliabilityAwareFusionNet(
    nn.Module
):

    def __init__(self):

        super().__init__()

        self.ecg = ModalityBranch(
            len(ECG_IDX)
        )

        self.bvp = ModalityBranch(
            len(BVP_IDX)
        )

        self.acc = ModalityBranch(
            len(ACC_IDX)
        )

        self.reliability = nn.Sequential(

            nn.Linear(
                len(ALL_FEATURE_COLS),
                32
            ),

            nn.ReLU(),

            nn.Linear(
                32,
                3
            )
        )


    def forward(
        self,
        x,
        return_weights=False
    ):

        ecg_logits = self.ecg(
            x[:, ECG_IDX]
        )

        bvp_logits = self.bvp(
            x[:, BVP_IDX]
        )

        acc_logits = self.acc(
            x[:, ACC_IDX]
        )

        reliability_logits = (
            self.reliability(x)
        )

        weights = torch.softmax(
            reliability_logits,
            dim=1
        )

        fused = (
            weights[:, 0:1]
            * ecg_logits
            +
            weights[:, 1:2]
            * bvp_logits
            +
            weights[:, 2:3]
            * acc_logits
        )

        if return_weights:
            return fused, weights

        return fused

In [ ]:
torch.manual_seed(SEED)

reliability_model = (
    ReliabilityAwareFusionNet()
)

reliability_model = train_model(
    reliability_model,
    train_loader,
    val_loader,
    epochs=100,
    lr=1e-3,
    patience=15
)

reliability_results = evaluate_model(
    reliability_model,
    test_loader,
    name="Reliability-Aware Fusion"
)

In [ ]:
clean_comparison = pd.DataFrame(
    [
        early_results,
        late_results,
        reliability_results
    ]
)

print(
    clean_comparison.to_string(
        index=False
    )
)

clean_comparison.to_csv(
    "/content/T1_2_clean_results.csv",
    index=False
)

In [ ]:
reliability_model.eval()

all_weights = []

with torch.no_grad():

    for xb, _ in test_loader:

        xb = xb.to(DEVICE)

        _, weights = reliability_model(
            xb,
            return_weights=True
        )

        all_weights.append(
            weights.cpu().numpy()
        )


all_weights = np.vstack(
    all_weights
)

print(
    "Mean ECG weight:",
    all_weights[:, 0].mean()
)

print(
    "Mean BVP weight:",
    all_weights[:, 1].mean()
)

print(
    "Mean ACC weight:",
    all_weights[:, 2].mean()
)

print("\nWeight SD:")

print(
    "ECG:",
    all_weights[:, 0].std()
)

print(
    "BVP:",
    all_weights[:, 1].std()
)

print(
    "ACC:",
    all_weights[:, 2].std()
)

In [ ]:
def get_predictions(model, X, y):

    model.eval()

    x = torch.tensor(
        X,
        dtype=torch.float32,
        device=DEVICE
    )

    with torch.no_grad():

        logits = model(x)

        probs = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        preds = logits.argmax(
            dim=1
        )

    return (
        preds.cpu().numpy(),
        probs.cpu().numpy()
    )


models = {
    "Early": early_model,
    "Late": late_model,
    "Reliability": reliability_model
}

for name, model in models.items():

    preds, probs = get_predictions(
        model,
        X_test,
        y_test
    )

    correctly_detected_stress = (
        (y_test == 1)
        & (preds == 1)
    ).sum()

    print(
        name,
        "| correctly detected stress:",
        correctly_detected_stress
    )

In [ ]:
def targeted_modality_pgd(
    model,
    x,
    modality_idx,
    epsilon,
    alpha,
    steps=20,
    target_class=0
):

    model.eval()

    x_original = x.detach().clone()

    x_adv = x_original.clone()

    # Random initialization ONLY
    # inside attacked modality
    noise = torch.zeros_like(x_adv)

    idx = torch.tensor(
        modality_idx,
        dtype=torch.long,
        device=x.device
    )

    noise[:, idx] = torch.empty(
        (
            x.shape[0],
            len(modality_idx)
        ),
        device=x.device
    ).uniform_(
        -epsilon,
        epsilon
    )

    x_adv = (
        x_adv + noise
    ).detach()

    target = torch.full(
        (x.shape[0],),
        target_class,
        dtype=torch.long,
        device=x.device
    )

    for _ in range(steps):

        x_adv.requires_grad_(True)

        logits = model(x_adv)

        loss = nn.CrossEntropyLoss()(
            logits,
            target
        )

        gradient = torch.autograd.grad(
            loss,
            x_adv
        )[0]

        updated = x_adv.detach().clone()

        # Targeted attack:
        # MINIMIZE target-class loss
        updated[:, idx] = (
            updated[:, idx]
            - alpha
            * gradient[:, idx].sign()
        )

        perturbation = (
            updated[:, idx]
            - x_original[:, idx]
        )

        perturbation = torch.clamp(
            perturbation,
            -epsilon,
            epsilon
        )

        updated[:, idx] = (
            x_original[:, idx]
            + perturbation
        )

        # Explicitly restore all non-attacked
        # features to their clean values.
        mask = torch.ones(
            x.shape[1],
            dtype=torch.bool,
            device=x.device
        )

        mask[idx] = False

        updated[:, mask] = (
            x_original[:, mask]
        )

        x_adv = updated.detach()

    return x_adv

In [ ]:
def evaluate_modality_attack(
    model,
    model_name,
    X,
    y,
    modality_name,
    modality_idx,
    epsilon,
    steps=20
):

    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32,
        device=DEVICE
    )

    y_tensor = torch.tensor(
        y,
        dtype=torch.long,
        device=DEVICE
    )

    # -------------------------
    # Clean predictions
    # -------------------------

    with torch.no_grad():

        clean_logits = model(
            X_tensor
        )

        clean_probs = torch.softmax(
            clean_logits,
            dim=1
        )[:, 1]

        clean_preds = clean_logits.argmax(
            dim=1
        )

    # Only attack true stress windows that
    # were correctly detected as stress.
    eligible_mask = (
        (y_tensor == 1)
        & (clean_preds == 1)
    )

    eligible_count = int(
        eligible_mask.sum().item()
    )

    if eligible_count == 0:
        return None

    X_eligible = X_tensor[
        eligible_mask
    ]

    # -------------------------
    # Targeted PGD
    # -------------------------

    alpha = epsilon / 4.0

    X_adv_eligible = targeted_modality_pgd(
        model=model,
        x=X_eligible,
        modality_idx=modality_idx,
        epsilon=epsilon,
        alpha=alpha,
        steps=steps,
        target_class=0
    )

    # Create full attacked test set:
    # only eligible windows are modified.
    X_adv_full = (
        X_tensor.detach().clone()
    )

    X_adv_full[
        eligible_mask
    ] = X_adv_eligible

    # -------------------------
    # Adversarial predictions
    # -------------------------

    with torch.no_grad():

        adv_logits = model(
            X_adv_full
        )

        adv_probs = torch.softmax(
            adv_logits,
            dim=1
        )[:, 1]

        adv_preds = adv_logits.argmax(
            dim=1
        )

    eligible_adv_preds = adv_preds[
        eligible_mask
    ]

    suppressed = int(
        (
            eligible_adv_preds == 0
        ).sum().item()
    )

    targeted_asr = (
        suppressed / eligible_count
    )

    # -------------------------
    # Perturbation magnitude
    # -------------------------

    idx = torch.tensor(
        modality_idx,
        dtype=torch.long,
        device=DEVICE
    )

    delta = (
        X_adv_eligible[:, idx]
        - X_eligible[:, idx]
    )

    linf = (
        delta.abs()
        .max(dim=1)
        .values
        .mean()
        .item()
    )

    l2 = (
        torch.norm(
            delta,
            p=2,
            dim=1
        )
        .mean()
        .item()
    )

    # Verify untouched modalities
    full_delta = (
        X_adv_full
        - X_tensor
    )

    attacked_set = set(
        modality_idx.tolist()
        if isinstance(
            modality_idx,
            np.ndarray
        )
        else modality_idx
    )

    other_idx = [
        i for i in range(
            X.shape[1]
        )
        if i not in attacked_set
    ]

    max_other_change = (
        full_delta[:, other_idx]
        .abs()
        .max()
        .item()
    )

    return {

        "Model":
            model_name,

        "Modality":
            modality_name,

        "Epsilon":
            epsilon,

        "Eligible":
            eligible_count,

        "Suppressed":
            suppressed,

        "Targeted_ASR":
            targeted_asr,

        "Accuracy":
            accuracy_score(
                y,
                adv_preds.cpu().numpy()
            ),

        "F1":
            f1_score(
                y,
                adv_preds.cpu().numpy(),
                zero_division=0
            ),

        "AUROC":
            roc_auc_score(
                y,
                adv_probs.cpu().numpy()
            ),

        "Mean_Linf":
            linf,

        "Mean_L2":
            l2,

        "Max_NonAttacked_Change":
            max_other_change
    }

In [ ]:
MODALITIES = {
    "ECG": ECG_IDX,
    "BVP": BVP_IDX,
    "ACC": ACC_IDX
}

EPSILONS = [
    0.01,
    0.025,
    0.05,
    0.10,
    0.20
]

attack_results = []

for model_name, model in models.items():

    for modality_name, modality_idx in MODALITIES.items():

        for epsilon in EPSILONS:

            print(
                f"{model_name:12s} | "
                f"{modality_name:3s} | "
                f"ε={epsilon:.3f}"
            )

            result = evaluate_modality_attack(
                model=model,
                model_name=model_name,
                X=X_test,
                y=y_test,
                modality_name=modality_name,
                modality_idx=modality_idx,
                epsilon=epsilon,
                steps=20
            )

            if result is not None:
                attack_results.append(
                    result
                )


attack_df = pd.DataFrame(
    attack_results
)

print(
    "\nConditions:",
    len(attack_df)
)

print(
    attack_df.to_string(
        index=False
    )
)

attack_df.to_csv(
    "/content/T1_2_single_modality_PGD.csv",
    index=False
)

In [ ]:
asr_table = attack_df.pivot_table(
    index=[
        "Model",
        "Epsilon"
    ],
    columns="Modality",
    values="Targeted_ASR"
).reset_index()

print(
    asr_table.to_string(
        index=False
    )
)

In [ ]:
auroc_table = attack_df.pivot_table(
    index=[
        "Model",
        "Epsilon"
    ],
    columns="Modality",
    values="AUROC"
).reset_index()

print(
    auroc_table.to_string(
        index=False
    )
)

In [ ]:
def get_reliability_weights(
    model,
    X
):

    model.eval()

    x = torch.tensor(
        X,
        dtype=torch.float32,
        device=DEVICE
    )

    with torch.no_grad():

        _, weights = model(
            x,
            return_weights=True
        )

    return weights.cpu().numpy()

In [ ]:
clean_weights = get_reliability_weights(
    reliability_model,
    X_test
)

weight_shift_rows = []

for modality_name, modality_idx in MODALITIES.items():

    X_tensor = torch.tensor(
        X_test,
        dtype=torch.float32,
        device=DEVICE
    )

    y_tensor = torch.tensor(
        y_test,
        dtype=torch.long,
        device=DEVICE
    )

    with torch.no_grad():

        clean_logits = reliability_model(
            X_tensor
        )

        clean_preds = clean_logits.argmax(
            dim=1
        )

    eligible_mask = (
        (y_tensor == 1)
        & (clean_preds == 1)
    )

    X_eligible = X_tensor[
        eligible_mask
    ]

    X_adv = targeted_modality_pgd(
        reliability_model,
        X_eligible,
        modality_idx,
        epsilon=0.10,
        alpha=0.025,
        steps=20,
        target_class=0
    )

    X_full_adv = X_tensor.clone()

    X_full_adv[
        eligible_mask
    ] = X_adv

    adv_weights = get_reliability_weights(
        reliability_model,
        X_full_adv.cpu().numpy()
    )

    weight_shift_rows.append({

        "Attacked_Modality":
            modality_name,

        "Clean_ECG_Weight":
            clean_weights[
                eligible_mask.cpu().numpy(),
                0
            ].mean(),

        "Adv_ECG_Weight":
            adv_weights[
                eligible_mask.cpu().numpy(),
                0
            ].mean(),

        "Clean_BVP_Weight":
            clean_weights[
                eligible_mask.cpu().numpy(),
                1
            ].mean(),

        "Adv_BVP_Weight":
            adv_weights[
                eligible_mask.cpu().numpy(),
                1
            ].mean(),

        "Clean_ACC_Weight":
            clean_weights[
                eligible_mask.cpu().numpy(),
                2
            ].mean(),

        "Adv_ACC_Weight":
            adv_weights[
                eligible_mask.cpu().numpy(),
                2
            ].mean()
    })


weight_shift_df = pd.DataFrame(
    weight_shift_rows
)

print(
    weight_shift_df.to_string(
        index=False
    )
)

In [ ]:
STRONG_EPSILONS = [
    0.20,
    0.30,
    0.50,
    0.75,
    1.00
]

strong_attack_results = []

for model_name, model in models.items():

    for modality_name, modality_idx in MODALITIES.items():

        for epsilon in STRONG_EPSILONS:

            print(
                f"{model_name:12s} | "
                f"{modality_name:3s} | "
                f"ε={epsilon:.2f}"
            )

            result = evaluate_modality_attack(
                model=model,
                model_name=model_name,
                X=X_test,
                y=y_test,
                modality_name=modality_name,
                modality_idx=modality_idx,
                epsilon=epsilon,
                steps=40
            )

            if result is not None:
                strong_attack_results.append(result)


strong_attack_df = pd.DataFrame(
    strong_attack_results
)

strong_asr = strong_attack_df.pivot_table(
    index=["Model", "Epsilon"],
    columns="Modality",
    values="Targeted_ASR"
).reset_index()

print("\nTARGETED ASR")
print(strong_asr.to_string(index=False))

strong_attack_df.to_csv(
    "/content/T1_2_strong_single_modality_PGD.csv",
    index=False
)

In [ ]:
def get_branch_stress_probs(
    model,
    X
):

    model.eval()

    x = torch.tensor(
        X,
        dtype=torch.float32,
        device=DEVICE
    )

    with torch.no_grad():

        ecg_logits = model.ecg(
            x[:, ECG_IDX]
        )

        bvp_logits = model.bvp(
            x[:, BVP_IDX]
        )

        acc_logits = model.acc(
            x[:, ACC_IDX]
        )

        ecg_prob = torch.softmax(
            ecg_logits, dim=1
        )[:, 1]

        bvp_prob = torch.softmax(
            bvp_logits, dim=1
        )[:, 1]

        acc_prob = torch.softmax(
            acc_logits, dim=1
        )[:, 1]

    return np.column_stack([
        ecg_prob.cpu().numpy(),
        bvp_prob.cpu().numpy(),
        acc_prob.cpu().numpy()
    ])

In [ ]:
clean_branch_probs = get_branch_stress_probs(
    late_model,
    X_test
)

for i, modality in enumerate(
    ["ECG", "BVP", "ACC"]
):

    branch_pred = (
        clean_branch_probs[:, i]
        >= 0.5
    ).astype(int)

    print("\n", modality)

    print(
        "Accuracy:",
        accuracy_score(
            y_test,
            branch_pred
        )
    )

    print(
        "F1:",
        f1_score(
            y_test,
            branch_pred,
            zero_division=0
        )
    )

    print(
        "AUROC:",
        roc_auc_score(
            y_test,
            clean_branch_probs[:, i]
        )
    )

In [ ]:
def attack_full_test_set(
    model,
    X,
    y,
    modality_idx,
    epsilon=0.5
):

    x = torch.tensor(
        X,
        dtype=torch.float32,
        device=DEVICE
    )

    y_tensor = torch.tensor(
        y,
        dtype=torch.long,
        device=DEVICE
    )

    with torch.no_grad():

        clean_logits = model(x)

        clean_preds = clean_logits.argmax(
            dim=1
        )

    eligible = (
        (y_tensor == 1)
        & (clean_preds == 1)
    )

    x_adv = x.clone()

    attacked = targeted_modality_pgd(
        model=model,
        x=x[eligible],
        modality_idx=modality_idx,
        epsilon=epsilon,
        alpha=epsilon / 8,
        steps=40,
        target_class=0
    )

    x_adv[eligible] = attacked

    return (
        x_adv.cpu().numpy(),
        eligible.cpu().numpy()
    )

In [ ]:
disagreement_rows = []

for attacked_name, attacked_idx in MODALITIES.items():

    X_adv, eligible = attack_full_test_set(
        late_model,
        X_test,
        y_test,
        attacked_idx,
        epsilon=0.5
    )

    clean_probs = get_branch_stress_probs(
        late_model,
        X_test
    )

    adv_probs = get_branch_stress_probs(
        late_model,
        X_adv
    )

    for i, branch_name in enumerate(
        ["ECG", "BVP", "ACC"]
    ):

        shift = np.abs(
            adv_probs[eligible, i]
            - clean_probs[eligible, i]
        )

        disagreement_rows.append({

            "Attacked_Modality":
                attacked_name,

            "Branch":
                branch_name,

            "Mean_Probability_Shift":
                shift.mean(),

            "Max_Probability_Shift":
                shift.max()
        })


disagreement_df = pd.DataFrame(
    disagreement_rows
)

print(
    disagreement_df.to_string(
        index=False
    )
)

In [ ]:
def late_fusion_excluding(
    model,
    X,
    exclude
):

    model.eval()

    x = torch.tensor(
        X,
        dtype=torch.float32,
        device=DEVICE
    )

    with torch.no_grad():

        logits = {
            "ECG": model.ecg(
                x[:, ECG_IDX]
            ),

            "BVP": model.bvp(
                x[:, BVP_IDX]
            ),

            "ACC": model.acc(
                x[:, ACC_IDX]
            )
        }

        kept = [
            value
            for name, value
            in logits.items()
            if name != exclude
        ]

        fused = torch.stack(
            kept
        ).mean(dim=0)

        probs = torch.softmax(
            fused,
            dim=1
        )[:, 1]

        preds = fused.argmax(
            dim=1
        )

    return (
        preds.cpu().numpy(),
        probs.cpu().numpy()
    )

In [ ]:
oracle_rows = []

for attacked_name, attacked_idx in MODALITIES.items():

    for epsilon in [
        0.2,
        0.5,
        1.0
    ]:

        X_adv, eligible = attack_full_test_set(
            late_model,
            X_test,
            y_test,
            attacked_idx,
            epsilon=epsilon
        )

        # Normal late fusion
        normal_preds, normal_probs = (
            get_predictions(
                late_model,
                X_adv,
                y_test
            )
        )

        normal_suppressed = (
            normal_preds[eligible] == 0
        ).sum()

        # Remove compromised modality
        defended_preds, defended_probs = (
            late_fusion_excluding(
                late_model,
                X_adv,
                exclude=attacked_name
            )
        )

        defended_suppressed = (
            defended_preds[eligible] == 0
        ).sum()

        n = eligible.sum()

        oracle_rows.append({

            "Attacked_Modality":
                attacked_name,

            "Epsilon":
                epsilon,

            "Eligible":
                n,

            "ASR_Normal":
                normal_suppressed / n,

            "ASR_After_Isolation":
                defended_suppressed / n,

            "AUROC_Normal":
                roc_auc_score(
                    y_test,
                    normal_probs
                ),

            "AUROC_After_Isolation":
                roc_auc_score(
                    y_test,
                    defended_probs
                )
        })


oracle_df = pd.DataFrame(
    oracle_rows
)

print(
    oracle_df.to_string(
        index=False
    )
)